In [24]:
%pip install langchain-chroma

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [28]:
from langchain_core.prompts import PromptTemplate
from langchain_chroma import Chroma
from langchain_community.llms import Ollama
from langchain_community.vectorstores import Chroma
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_huggingface import HuggingFaceEmbeddings

In [32]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma(
    persist_directory="chroma_db",
    embedding_function=embeddings,
    collection_name="langchain"
)

template = """You are an expert coding assistant. Use the following documentation excerpts to answer the question accurately. If the answer is not in the context, say "I don't know."

Context:
{context}

Question: {question}

Answer:"""

prompt = PromptTemplate(
    input_variables=["context", "question"],
    template=template
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6966.50it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [33]:
llm = Ollama(model="llama3", temperature=0.1)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
rag_chain = (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )

In [34]:
question = "How do I use dictionary in python?"
answer = rag_chain.invoke(question)
source_docs = retriever.invoke(question)
print("=== ASSISTANT'S ANSWER ===")
print(answer)
    
print("\n=== SOURCES USED ===")
for idx, doc in enumerate(source_docs):
    source_file = doc.metadata.get("source", "Unknown file")
    print(f"{idx + 1}. {source_file}")

=== ASSISTANT'S ANSWER ===
I can help you with that!

According to the documentation, Python's dictionaries are ordered by default since Python 3.7, so you can use them as is. There is no specific function or method to use dictionaries in Python, as they are a built-in data structure.

Here's a simple example of how you can use a dictionary in Python:

```
my_dict = {'name': 'John', 'age': 30}
print(my_dict['name'])  # Output: John
print(my_dict['age'])   # Output: 30
```

In this example, we create a dictionary called `my_dict` with two key-value pairs: `name` with the value `'John'` and `age` with the value `30`. Then, we access the values using their corresponding keys.

=== SOURCES USED ===
1. ..\..\docs\dict.html
2. ..\..\docs\dict.html
3. ..\..\docs\dict.html
